# 00 – Download Remote Sensing Data
**Purpose:** Download different datasets, which will be used in the following notebooks. To Download all the datasets around **.... GB/MB** of space are needed.

| Dataset | Working-Name | Resolution | Time-range | API | URL |
| --- | --- | --- | --- | --- |--- |
|SWOT_L2_HR_RiverSP_reach_D || global, but need bbox  | Dec 2022 - | Yes |https://swot.jpl.nasa.gov https://podaac.jpl.nasa.gov/dataset/SWOT_L2_HR_Raster_D|
| ESA WorldCover ||  global but bbox, 10m | 2020 - 2021 |Yes |https://esa-worldcover.org/en|
|SoilGrids||global, 250m|||https://docs.isric.org/globaldata/soilgrids/|
|DEM||global, 30m||||
|GlobalDamWatch||global||||
|Global River Classification (GloRiC)||||||
|||||||

**Workflow:**
0 | Imports
- 0.1 | Configuration
- 0.2 | Hydrocron API (no login required)
- 0.3 | earthaccess (NASA login required)

**References:**
- SWOT Mission: https://swot.jpl.nasa.gov
- Hydrocron API: https://github.com/podaac/hydrocron
- PO.DAAC SWOT Data: (https://podaac.jpl.nasa.gov/SWOT)
- Tebaldi, N., Nickels, C. (2025): [*Hydrocron API: SWOT Time Series Examples.*](https://podaac.github.io/tutorials/notebooks/datasets/Hydrocron_SWOT_timeseries_examples.html)
- Nickels, C. (2025): [*Search and Download SWOT Data via <code>earthaccess</code>*. ](https://podaac.github.io/tutorials/notebooks/SearchDownload_SWOTviaCMR.html)
- earthaccess docs: (https://earthaccess.readthedocs.io)
- NASA Earthdata Login: (https://urs.earthdata.nasa.gov)

To-Do:
- [] Thinking about variables which can be calculated by SWOT data and which are not in SWORD dataset
- [] making the workflow for investigation and download and variable calculation before download smarter.
- [] do I even need to download? I could just ask API for values, calculate new ones and join them to the SWORD dataset?
- [] How should can I besser handle the amount of AWOT data? Maybe...by..maybe creating a Goal-Reach_list and using parquet instead of shp and gpkg.
- [] adding Klimaclassification köppen & Geiger
- [] Connectivity Index in River Networks für die GDW (aufbauend, also pro z.B. Damm immer nur Oberfluss)
- [] Wegen GDW: Schauen, ob sich Messwerte in SWORD nodes signifikant an der Stelle ändern, an welcher das GDW Objekt auftritt.
- [] connectivity flussaufwärts mit GDW bauwerke 
- [] SWORD reaches und HydroBasins download ergänzen

---
---
## 0 | Imports


In [ ]:
import geopandas as gpd
import os
import sys
import earthaccess
import rasterio
import zipfile
import requests
from pathlib import Path
from soilgrids import soilgrids

sys.path.append(os.path.join("..", "src"))

from _0_config_paths import DATA_RAW, DATA_PROCESSED, DATA_OUTPUT, IN_SWORD
from _00_config import STUDY_AREA, PFAF_IDS
from _00_setup import setup_study_area
from _0_config_paths import IN_BASIN_5
from _00_config import PFAF_IDS, PFAF_LEVEL_DIGITS

from _00_download_rs_data import (
    query_hydrocron_bulk,
    parse_hydrocron_results,
    search_swot_granules,
    stream_swot_granules,
    download_soilgrids,
    download_worldcover,
    merge_raster_tiles,
    #download_opentopo_dem,
    download_cop30_aws,
    download_gloric_v10,
    download_gdw
)

SAGA found: C:\Program Files\QGIS 3.42.3\apps\saga7\saga_cmd.exe


---
---
## 0.1 | Configuration

In [2]:
# ============================================================
# Study area is defined in src/_00_config.py
# Only download-specific settings are configured here
# ============================================================

# Load AOI and SWORD from study area setup
aoi, sword = setup_study_area(
    basin_path        = IN_BASIN_5,
    sword_path        = IN_SWORD,
    pfaf_ids          = PFAF_IDS,
    pfaf_level_digits = PFAF_LEVEL_DIGITS
)
bbox = tuple(aoi.total_bounds)
print(f"AOI bbox: {bbox}")

#--- 0.2 HYDROCRON settings -----------------------------------------------
HYDROCRON_FIELDS        = "reach_id,time_str,wse,slope,width,p_dist_out,geometry"
HYDROCRON_OUTPUT_FORMAT = "geojson"
HYDROCRON_REQUEST_DELAY = 0.3
MIN_WIDTH_M             = 100

#--- 0.2 + 0.3 TIME RANGE for SWOT observations --------------------------------
START_TIME = "2025-08-01T00:00:00Z"
END_TIME   = "2025-08-31T00:00:00Z"

#--- 0.3 EARTHACCESS settings ---------------------------------------------
EARTHACCESS_COLLECTION = "SWOT_L2_HR_RiverSP_reach_D"
GRANULE_STRIDE         = 1
MAX_GRANULES           = None

#--- 0.5 SoilGrids -------------------------------------------------------
SOILGRIDS_VARS = [
    ("clay", "clay_0-5cm_mean", "Clay content (%) – cohesive banks"),
    ("sand", "sand_0-5cm_mean", "Sand content (%) – non-cohesive banks"),
    ("silt", "silt_0-5cm_mean", "Silt content (%) – intermediate grain size"),
    ("cfvo", "cfvo_0-5cm_mean", "Coarse fragments (%) – rocky soils"),
]

#--- 0.6 OpenTopography API key (set as environment variable, never hardcode)----
OPENTOPO_API_KEY = os.environ.get("OPENTOPO_API_KEY", "")

#--- OUTPUT paths -------------------------------------------------------
OUT_HYDROCRON     = os.path.join(DATA_RAW, f"swot_hydrocron_{STUDY_AREA}.gpkg")
OUT_EARTHACCESS   = os.path.join(DATA_RAW, f"swot_earthaccess_{STUDY_AREA}.gpkg")
OUT_WORLDCOVER_DIR = os.path.join(DATA_RAW, "worldcover")
OUT_WORLDCOVER_MERGED = os.path.join(DATA_RAW, "worldcover", "worldcover_merged.tif")
OUT_SOILGRIDS_DIR = os.path.join(DATA_RAW, "soilgrids")
OUT_COP30_DEM     = os.path.join(DATA_RAW,  f"{STUDY_AREA}_example",
                                  f"{STUDY_AREA}_cop30_dem.tif")
OUT_GDW_DIR       = os.path.join(DATA_RAW, "GDW")
OUT_GLORIC_V10    = os.path.join(DATA_RAW, "GloRiC_v10.gpkg")


print(f"Study area : {STUDY_AREA}")
print(f"PFAF_IDs   : {PFAF_IDS}")
print(f"bbox       : {bbox}")

Loading HydroBASINS Level 5 from: D:\0_InnoLab\0_data\BasinATLAS_HydroSHEDS\BasinATLAS_v10_lev05.gpkg
Basin polygons found: 1 for PFAF_IDs: [46219]
AOI bbox: [71.3625     40.4375     78.35416667 42.46666667]

Loading SWORD from: D:\0_InnoLab\0_data\SWOT_river_database_SWORD\as_sword_reaches_v17b.gpkg
SWORD reaches in bbox: 214
SWORD reaches after PFAF_ID filter: 140
AOI bbox: (np.float64(71.36249999957539), np.float64(40.43750000022487), np.float64(78.35416666695818), np.float64(42.46666666673332))
Study area : naryn
PFAF_IDs   : [46219]
bbox       : (np.float64(71.36249999957539), np.float64(40.43750000022487), np.float64(78.35416666695818), np.float64(42.46666666673332))


---
---
## 0.2 | Hydrocron API

Queries SWOT time series per SWORD reach. No file download, no NASA login required.
SWOT observes rivers wider than ca. 100 m, narrower reaches are skipped automatically.

Needing the **Hydrocron API** (PO.DAAC / NASA JPL) for each SWORD reach in the AOI.
Returns all SWOT observations within the configured time range as GeoJSON or CSV.

**How:** SWOT data is archived as globally-tiled shapefiles per satellite pass
(one file = one pass over a whole continent). Hydrocron unpacks these shapefiles and
re-indexes them by reach_id, so a single API call returns all observations for one reach
across all passes --> without downloading any file. **No NASA login required.**

https://github.com/podaac/hydrocron

In [3]:
print(f"SWORD reaches loaded: {len(sword)} | CRS: {sword.crs}")

# Query all reaches via Hydrocron API
all_features = query_hydrocron_bulk(
    sword = sword,
    start_time= START_TIME,
    end_time = END_TIME,
    fields = HYDROCRON_FIELDS,
    min_width_m = MIN_WIDTH_M,
    request_delay = HYDROCRON_REQUEST_DELAY
)

SWORD reaches loaded: 140 | CRS: EPSG:4326
Reaches queried (width >= 100m): 47
Skipped (too narrow): 93
Time range: 2025-08-01T00:00:00Z → 2025-08-31T00:00:00Z
  0/47 | ok=0 no_data=0 error=0
 10/47 | ok=10 no_data=0 error=0
 20/47 | ok=20 no_data=0 error=0
 30/47 | ok=30 no_data=0 error=0
 40/47 | ok=40 no_data=0 error=0

Done. ok=46 | no_data=0 | errors=1
Total observations collected: 199

Error log (first 10):
  reach 46219900396: 400 Bad Request: {'error': '400: Results with the specified Feature ID 46219900396 were not found'}


In [4]:
# Parse raw GeoJSON into clean GeoDataFrame and save
swot_gdf = parse_hydrocron_results(all_features)

if swot_gdf is not None:
    num_cols = [c for c in ["wse", "slope", "width"] if c in swot_gdf.columns]
    print(swot_gdf[num_cols].describe().round(3))
    print(f"\nObservations per reach:")
    print(swot_gdf.groupby("reach_id").size().describe())

    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_gdf.to_file(OUT_HYDROCRON, driver="GPKG")
    print(f"\nSaved: {OUT_HYDROCRON}")

Observations parsed: 199
Columns: ['geometry', 'reach_id', 'time_str', 'wse', 'slope', 'width', 'p_dist_out', 'wse_units', 'slope_units', 'width_units', 'p_dist_out_units']
            wse   slope     width
count   148.000  94.000    98.000
mean   1349.420   0.004   402.714
std     786.592   0.002   438.221
min     403.200  -0.002     0.000
25%     723.699   0.003   181.000
50%    1258.199   0.003   272.837
75%    1661.500   0.004   472.918
max    3741.613   0.012  3082.287

Observations per reach:
count    46.000000
mean      4.326087
std       0.700931
min       3.000000
25%       4.000000
50%       4.000000
75%       5.000000
max       6.000000
dtype: float64

Saved: D:\0_InnoLab\0_data\swot_hydrocron_naryn.gpkg


c:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\notebooks\..\src\_00_download_rs_data.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdf["time_str"] = pd.to_datetime(gdf["time_str"], errors="coerce", utc=True)


---
---
## 0.3 | earthaccess – Download SWOT granules

Downloads the actual SWOT Shapefile granules (one file = one satellite pass over a
continent). Files are filtered spatially by bounding box and temporally by date range.

**downloadable in this section:**
- Raster products (`L2_HR_Raster`)
- Full Shapefile attributes (not just the Hydrocron field subset)

**!!!Warning:!!!** Each SWOT granule covers an entire continent pass and can be **several
hundred MB to several GB**. After download the data is clipped to the AOI.

**NASA Earthdata Login required.** acc at: https://urs.earthdata.nasa.gov

References:
- earthaccess: https://earthaccess.readthedocs.io
- SWOT collections: https://podaac.github.io/tutorials/quarto_text/SWOT.html
- PO.DAAC Cookbook: https://podaac.github.io/tutorials

In [5]:
# EARTHACCESS: authenticate with NASA Earthdata Login

# earthaccess.login() will:
#   Try to use a stored .netrc file silently
#   If not found: prompt for username + password interactively

auth = earthaccess.login(strategy="interactive", persist=True)
print(f"Login successful: {auth.authenticated}")

Login successful: True


In [6]:
# Search for matching granules
results = search_swot_granules(
    collection = EARTHACCESS_COLLECTION,
    bbox = bbox,
    start_time = START_TIME,
    end_time = END_TIME
)

# Stream granules in-memory, clip to AOI, merge into GeoDataFrame
swot_ea = stream_swot_granules(
    results = results,
    aoi_bounds = sword.total_bounds,
    max_granules = MAX_GRANULES,
    granule_stride = GRANULE_STRIDE
)

if swot_ea is not None:
    os.makedirs(DATA_PROCESSED, exist_ok=True)
    swot_ea.to_file(OUT_EARTHACCESS, driver="GPKG")
    print(f"Saved: {OUT_EARTHACCESS}")

c:\Users\Duck\anaconda3\envs\river_egg\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


First: 2025-08-01T05:35:12.755Z
Last : 2025-08-27T02:27:02.793Z
Granules found: 42

Estimated download size (if all downloaded): ca. 3 GB
Granules to process : 42 of 42
Stride: 1
AOI bounds: [71.7547 40.9007 78.2482 42.3158]
   0/42 | ok=0 empty=0 error=0


KeyboardInterrupt: 

---
## 0.4 | Download ESA WorldCover

In [7]:
worldcover_files = download_worldcover(
    bbox    = bbox,
    out_dir = OUT_WORLDCOVER_DIR
)
print(f"WorldCover tiles: {worldcover_files}")

Tiles needed: 8
  lon=69, lat=39
  lon=69, lat=42
  lon=72, lat=39
  lon=72, lat=42
  lon=75, lat=39
  lon=75, lat=42
  lon=78, lat=39
  lon=78, lat=42
Downloading: ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
  Saved: D:\0_InnoLab\0_data\worldcover\ESA_WorldCover_10m_2021_v200_N39E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
  Saved: D:\0_InnoLab\0_data\worldcover\ESA_WorldCover_10m_2021_v200_N42E069_Map.tif
Downloading: ESA_WorldCover_10m_2021_v200_N39E072_Map.tif


KeyboardInterrupt: 

In [ ]:
worldcover_merged = merge_raster_tiles(
    tile_paths = worldcover_files,
    out_path   = OUT_WORLDCOVER_MERGED
)

Merging 8 tiles...
Merged 8 tiles → D:\0_InnoLab\0_data\WorldCover\WorldCover_merged.tif


---
## 0.5 | Download soilgrids


In [ ]:
soilgrids_files = download_soilgrids(
    bbox           = bbox,
    out_dir        = OUT_SOILGRIDS_DIR,
    soilgrids_vars = SOILGRIDS_VARS
)
print(f"soilgrids files: {soilgrids_files}")

Requesting grid: 2927 x 671 pixels
Downloading: clay_0-5cm_mean – Clay content (%) – cohesive banks, meandering tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\clay_0-5cm_mean.tif
Downloading: sand_0-5cm_mean – Sand content (%) – non-cohesive banks, braided tendency
  Saved: D:\0_InnoLab\0_data\SoilGrids\sand_0-5cm_mean.tif
Downloading: silt_0-5cm_mean – Silt content (%) – intermediate grain size
  Saved: D:\0_InnoLab\0_data\SoilGrids\silt_0-5cm_mean.tif
Downloading: cfvo_0-5cm_mean – Coarse fragments volume (%) – rocky soils, mountain reaches
  Saved: D:\0_InnoLab\0_data\SoilGrids\cfvo_0-5cm_mean.tif

Download complete: 4 variables
SoilGrids files: ['D:\\0_InnoLab\\0_data\\SoilGrids\\clay_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\sand_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\silt_0-5cm_mean.tif', 'D:\\0_InnoLab\\0_data\\SoilGrids\\cfvo_0-5cm_mean.tif']


---
## 06 | Download DEM 

In [18]:
dem_path= download_cop30_aws(bbox = bbox, out_path= OUT_COP30_DEM)

Downloaded: Copernicus_DSM_COG_10_N40_00_E071_00_DEM/Copernicus_DSM_COG_10_N40_00_E071_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E072_00_DEM/Copernicus_DSM_COG_10_N40_00_E072_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E073_00_DEM/Copernicus_DSM_COG_10_N40_00_E073_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E074_00_DEM/Copernicus_DSM_COG_10_N40_00_E074_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E075_00_DEM/Copernicus_DSM_COG_10_N40_00_E075_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E076_00_DEM/Copernicus_DSM_COG_10_N40_00_E076_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E077_00_DEM/Copernicus_DSM_COG_10_N40_00_E077_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N40_00_E078_00_DEM/Copernicus_DSM_COG_10_N40_00_E078_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N41_00_E071_00_DEM/Copernicus_DSM_COG_10_N41_00_E071_00_DEM.tif
Downloaded: Copernicus_DSM_COG_10_N41_00_E072_00_DEM/Copernicus_DSM_COG_10_N41_00_E072_00_DEM.tif
Downloaded: Copernic

In [19]:
print(OUT_COP30_DEM)

D:\0_InnoLab\0_data\naryn_example\naryn_cop30_dem.tif


---
## 07 | Download GlobalDamWatch (GDW)

Download the global dataset once.

In [20]:
gdw_dir = download_gdw(out_dir=OUT_GDW_DIR)
print(f"GDW ready at: {gdw_dir}")

Extracted 17 files to: D:\0_InnoLab\0_data\GDW
GDW ready at: D:\0_InnoLab\0_data\GDW


---
## 08 | Download Global River Classification (GloRiC)

https://www.hydrosheds.org/products/gloric

In [22]:
gloric_gpkg = download_gloric_v10(gpkg_path=OUT_GLORIC_V10)
print(f"GloRiC v1.0 ready at: {gloric_gpkg}")

Extracting ZIP...
Extracted 9 files to: D:\0_InnoLab\0_data\GloRiC_v10_shapefile
Found shapefiles:
 - D:\0_InnoLab\0_data\GloRiC_v10_shapefile\GloRiC_v10_shapefile\GloRiC_v10.shp
Using shapefile: D:\0_InnoLab\0_data\GloRiC_v10_shapefile\GloRiC_v10_shapefile\GloRiC_v10.shp
Reading shapefile: D:\0_InnoLab\0_data\GloRiC_v10_shapefile\GloRiC_v10_shapefile\GloRiC_v10.shp
Saving GeoPackage: D:\0_InnoLab\0_data\GloRiC_v10.gpkg
Done.
GloRiC v1.0 ready at: D:\0_InnoLab\0_data\GloRiC_v10.gpkg
